# Train Commutative Transformer Classification Heads

Load the commutative transformer encoder saved by notebook 12T, freeze the encoder/backbone, and train only the supervised classification heads on the current labeled action dataset.

In [ ]:
%load_ext autoreload
%autoreload 2

from dataclasses import asdict
from pathlib import Path

import pandas as pd

from src.ml import (
    CommutativeTransformerClassifier,
    LossWeightConfig,
    OptimizationConfig,
    display_experiment_summary,
    display_holdout_evaluation,
    create_experiment_run,
    fit_estimator_on_experiment,
    load_commutative_transformer_pretraining_config,
    persist_experiment_artifacts,
    plot_training_history,
    prepare_multitask_experiment_data,
)
from src.dataset_config import load_current_dataset_artifact_path
from src.tensor_utils import build_tensor_embedding_2d, load_labeled_tensor_dataset, plot_tensor_embedding_2d

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)

In [ ]:
# User inputs

dataset_artifact_path = load_current_dataset_artifact_path()
pretraining_config_path = Path("artifacts/pretrained_commutative_transformer/config.yaml")
pretraining_config = load_commutative_transformer_pretraining_config(pretraining_config_path)
print(f"Loaded commutative transformer pretraining config from {pretraining_config_path}")
print(pretraining_config)

pretrained_encoder_path = pretraining_config.pretrained_encoder_path
model_config = pretraining_config.model_config
experiment_output_dir = Path("artifacts/nb14T_commutative_transformer_head_only")
experiment_run = create_experiment_run(experiment_output_dir, "14T_train_commutative_transformer_head")
loss_plot_dir = Path(experiment_run.loss_plot_dir) / "training"
figure_dir = Path(experiment_run.figure_dir)
persist_artifacts = True
print(f"Experiment id: {experiment_run.experiment_id}")
print(f"Experiment run folder: {Path(experiment_run.run_dir).resolve()}")
print(f"Training loss PDFs: {loss_plot_dir.resolve()}")

holdout_fraction = 0.25
validation_fraction_within_train = 0.20
train_num_random_rotations = 6
rotation_range_degrees = 12.0
freeze_backbone = True

optimization_config = OptimizationConfig(
    batch_size=8,
    epochs=100,
    learning_rate=5e-5,
    weight_decay=3e-3,
    early_stopping_patience=8,
    early_stopping_min_delta=0.0,
    training_plot_dir=str(loss_plot_dir),
    training_plot_every_n_epochs=1,
    scheduler_patience=2,
    scheduler_factor=0.7,
    scheduler_min_lr=1e-6,
    validation_split=0.0,
    random_state=0,
    standardize=True,
    device=None,
    verbose=True,
)
loss_weight_config = LossWeightConfig(
    action_weight=1.0,
    compound_weight=0.05,
    concentration_weight=0.05,
    lambda_align=0.0,
)

In [ ]:
if not pretrained_encoder_path.exists():
    raise FileNotFoundError(
        f"Pretrained transformer encoder not found at {pretrained_encoder_path}. "
        "Run notebook 12T first."
    )

dataset = load_labeled_tensor_dataset(dataset_artifact_path)
experiment = prepare_multitask_experiment_data(
    dataset,
    holdout_fraction=holdout_fraction,
    validation_fraction_within_train=validation_fraction_within_train,
    train_num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    random_state=optimization_config.random_state,
)
display_experiment_summary(experiment)

In [ ]:
model = CommutativeTransformerClassifier(
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
    pretrained_state_path=pretrained_encoder_path,
    freeze_backbone=freeze_backbone,
)
fit_estimator_on_experiment(model, experiment)
print(f"Writing training loss PDFs to: {loss_plot_dir.resolve()}")
plot_training_history(model, title="Pretrained commutative transformer head-only loss curves", loess_frac=0.6);

In [ ]:
holdout_evaluation = display_holdout_evaluation(model, experiment)

In [ ]:
holdout_embedding_projection = build_tensor_embedding_2d(
    model.transform(experiment.splits.X_holdout),
    experiment.y_true_holdout["action"],
    label_map=experiment.label_maps["action"],
    metadata=experiment.splits.metadata_holdout,
    method="umap",
    random_state=optimization_config.random_state,
)
holdout_embedding_projection.to_csv(
    figure_dir / f"{experiment_run.experiment_id}_holdout_embedding_umap.csv",
    index=False,
)
plot_tensor_embedding_2d(
    holdout_embedding_projection,
    title="Holdout embedding projection by action",
    marker_column="compound",
    output_path=figure_dir / f"{experiment_run.experiment_id}_holdout_embedding_umap.pdf",
)

In [ ]:
run_config = {
    "experiment_id": experiment_run.experiment_id,
    "experiment_run_dir": Path(experiment_run.run_dir),
    "dataset_artifact_path": dataset_artifact_path,
    "pretraining_config_path": pretraining_config_path,
    "pretrained_encoder_path": pretrained_encoder_path,
    "freeze_backbone": freeze_backbone,
    "holdout_fraction": holdout_fraction,
    "validation_fraction_within_train": validation_fraction_within_train,
    "train_num_random_rotations": train_num_random_rotations,
    "rotation_range_degrees": rotation_range_degrees,
    "model_config": asdict(model_config),
    "optimization_config": asdict(optimization_config),
    "loss_weight_config": asdict(loss_weight_config),
}
if persist_artifacts:
    experiment_artifacts = persist_experiment_artifacts(
        output_dir=experiment_output_dir,
        estimator=model,
        reports=holdout_evaluation.reports,
        config=run_config,
        experiment_prefix="14T_train_commutative_transformer_head",
        experiment_id=experiment_run.experiment_id,
        evaluation=holdout_evaluation,
        experiment=experiment,
        loss_plot_dirs=[loss_plot_dir],
    )
    experiment_artifacts